In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: una Qiskit Function di Multiverse Computing
> **Note:** * Le Qiskit Functions sono una funzionalità sperimentale disponibile solo per gli utenti dei piani IBM Quantum&reg; Premium, Flex e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Con la funzione "Singularity Machine Learning - Classification", puoi risolvere problemi reali di machine learning su hardware quantistico senza necessitare di competenze specifiche in ambito quantistico. Questa funzione applicativa, basata su metodi ensemble, è un classificatore ibrido. Sfrutta metodi classici come boosting, bagging e stacking per l'addestramento iniziale dell'ensemble. Successivamente, algoritmi quantistici come il variational quantum eigensolver (VQE) e il quantum approximate optimization algorithm (QAOA) vengono impiegati per migliorare la diversità, le capacità di generalizzazione e la complessità complessiva dell'ensemble addestrato.

A differenza di altre soluzioni di machine learning quantistico, questa funzione è in grado di gestire dataset su larga scala con milioni di esempi e feature senza essere limitata dal numero di Qubit nel QPU di destinazione. Il numero di Qubit determina soltanto la dimensione dell'ensemble che può essere addestrato. È anche molto flessibile e può essere utilizzata per risolvere problemi di classificazione in un'ampia gamma di domini, tra cui finanza, sanità e cybersicurezza.
Raggiunge costantemente alte accuratezze su problemi classicamente difficili che coinvolgono dataset ad alta dimensionalità, rumorosi e sbilanciati.
![Come funziona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
È pensata per:
1. Ingegneri e data scientist di aziende che desiderano potenziare la propria offerta tecnologica integrando il machine learning quantistico nei propri prodotti e servizi,
2. Ricercatori di laboratori di ricerca quantistica che esplorano applicazioni di machine learning quantistico e vogliono sfruttare il calcolo quantistico per attività di classificazione, e
3. Studenti e docenti di istituzioni educative in corsi come quello di machine learning, che desiderano dimostrare i vantaggi del calcolo quantistico.

Il seguente esempio ne mostra le varie funzionalità, tra cui `create`, `list`, `fit` e `predict`, e ne dimostra l'utilizzo su un problema sintetico composto da due semicerchi a forma di mezza luna che si intrecciano, un problema notoriamente difficile a causa del suo confine decisionale non lineare.


## Descrizione della funzione
Questa Qiskit Function permette agli utenti di risolvere problemi di classificazione binaria utilizzando il classificatore ensemble potenziato quantisticamente di Singularity. Dietro le quinte, usa un approccio ibrido per addestrare classicamente un ensemble di classificatori sul dataset etichettato, e poi lo ottimizza per massimizzare la diversità e la generalizzazione usando il Quantum Approximate Optimization Algorithm (QAOA) sui QPU IBM&reg;. Tramite un'interfaccia intuitiva, gli utenti possono configurare un classificatore in base alle proprie esigenze, addestrarlo sul dataset desiderato e usarlo per fare previsioni su un dataset mai visto in precedenza.

Per risolvere un generico problema di classificazione:
1. Preelabora il dataset e suddividilo in set di addestramento e di test. Facoltativamente, puoi suddividere ulteriormente il set di addestramento in set di addestramento e di validazione. Questo può essere ottenuto usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Se il set di addestramento è sbilanciato, puoi ricampionarlo per bilanciare le classi usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Carica separatamente i set di addestramento, validazione e test nell'archiviazione della funzione usando il metodo `file_upload` del catalogo, passando il percorso corrispondente ogni volta.
4. Inizializza il classificatore quantistico usando l'azione `create` della funzione, che accetta iperparametri come il numero e i tipi di learner, la regolarizzazione (valore lambda) e le opzioni di ottimizzazione inclusi il numero di livelli, il tipo di ottimizzatore classico, il backend quantistico e così via.
5. Addestra il classificatore quantistico sul set di addestramento usando l'azione `fit` della funzione, passandogli il set di addestramento etichettato e il set di validazione se applicabile.
6. Esegui previsioni sul set di test precedentemente non visto usando l'azione `predict` della funzione.

## Approccio basato su azioni
La funzione utilizza un approccio basato su azioni. Puoi pensarla come un ambiente virtuale in cui usi le azioni per eseguire attività o modificarne lo stato. Attualmente, offre le seguenti azioni: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) e [create_fit_predict](#7-create-fit-predict). L'esempio seguente illustra l'azione `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

L'azione `list` recupera tutti i classificatori memorizzati in formato `*.pkl.tar` dalla directory dati condivisa. Puoi anche accedere al contenuto di questa directory usando il metodo `catalog.files()`. In generale, l'azione list cerca file con l'estensione `*.pkl.tar` nella directory dati condivisa e li restituisce in formato lista.

#### Input

|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | Il nome dell'azione tra `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sì |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
L'azione `create` crea un classificatore del tipo `quantum_classifier` specificato usando i parametri forniti, e lo salva nella directory dati condivisa.

> **Note:** La funzione attualmente supporta solo `QuantumEnhancedEnsembleClassifier`.

#### Input
|     Nome    |    Tipo     | Descrizione | Obbligatorio  | Predefinito |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | Il nome dell'azione tra `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sì | - |
| `name` | `str` | Il nome del classificatore quantistico, ad es. `spam_classifier`. | Sì | - |
| `instance` | `str` | Istanza IBM. | Sì | - |
| `backend_name` | `str` | Risorsa di calcolo IBM. Il valore predefinito è `None`, il che significa che verrà usato il backend con il minor numero di job in coda. | No | `None` |
| `quantum_classifier` | `str` | Il tipo di classificatore quantistico, ovvero `QuantumEnhancedEnsembleClassifier`. | No | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Il numero di learner nell'ensemble. | No | `10` |
| `learners_types` | `list` | Tipi di learner. Tra i tipi supportati vi sono: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` e `LogisticRegression`. Ulteriori dettagli su ciascuno sono disponibili nella [documentazione di scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | No | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporzioni di ciascun tipo di learner nell'ensemble. | No | `[1.0]` |
| `learners_options` | `list` | Opzioni per ciascun tipo di learner nell'ensemble. Per un elenco completo delle opzioni corrispondenti al tipo o ai tipi di learner scelti, consulta la [documentazione di scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | No | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` o `list` | Tipo/i di regolarizzazione da utilizzare: `onsite` o `alpha`. `onsite` controlla il termine onsite dove valori più alti portano a ensemble più sparsi. `alpha` controlla il compromesso tra termini di interazione e onsite dove valori più bassi portano a ensemble più sparsi. Se viene fornita una lista, verranno addestrati modelli per ciascun tipo e verrà selezionato quello con le prestazioni migliori. | No | `onsite` |
| `regularization` | `str` o `float` o `list` | Valore di regolarizzazione. Compreso tra `0` e `+inf` se regularization_type è `onsite`. Compreso tra `0` e `1` se regularization_type è `alpha`. Se impostato su `auto`, viene usata la regolarizzazione automatica: il parametro di regolarizzazione ottimale viene trovato tramite ricerca binaria con il rapporto desiderato di classificatori selezionati sul totale (`regularization_desired_ratio`) e il limite superiore per il parametro di regolarizzazione (`regularization_upper_bound`). Se viene fornita una lista, verranno addestrati modelli per ciascun valore e verrà selezionato quello con le prestazioni migliori. | No | `0.01` |
| `regularization_desired_ratio` | `float` o `list` | Rapporto/i desiderato/i di classificatori selezionati sul totale per la regolarizzazione automatica. Se viene fornita una lista, verranno addestrati modelli per ciascun rapporto e verrà selezionato quello con le prestazioni migliori. | No | `0.75` |
| `regularization_upper_bound` | `float` o `list` | Limite/i superiore/i per il parametro di regolarizzazione quando si usa la regolarizzazione automatica. Se viene fornita una lista, verranno addestrati modelli per ciascun limite superiore e verrà selezionato quello con le prestazioni migliori. | No | `200` |
| `weight_update_method` | `str` | Metodo per l'aggiornamento dei pesi dei campioni tra `logarithmic` e `quadratic`. | No | `logarithmic` |
| `sample_scaling` | `boolean` | Se applicare il ridimensionamento dei campioni. | No | `False` |
| `prediction_scaling` | `float` | Fattore di scala per le previsioni. | No | `None` |
| `optimizer_options` | `dictionary` | Opzioni dell'ottimizzatore QAOA. Un elenco delle opzioni disponibili è presentato più avanti in questa documentazione. | No | ... |
| `voting` | `str` | Usa il voto di maggioranza (`hard`) o la media delle probabilità (`soft`) per aggregare le previsioni/probabilità dei learner. | No | `hard` |
| `prob_threshold` | `float` | Soglia di probabilità ottimale. | No | `0.5` |
| `random_state` | `integer` | Controlla la casualità per la riproducibilità. | No | `None` |


- Inoltre, le `optimizer_options` sono elencate come segue:

|     Nome    |    Tipo     | Descrizione | Obbligatorio  | Predefinito |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | Il numero di soluzioni | No | `1024` |
| `reps` | `integer` | Il numero di ripetizioni | No | `4` |
| `sparsify` | `float` | La soglia di sparsificazione | No | `0.001` |
| `theta` | `float` | Il valore iniziale di theta, un parametro variazionale di QAOA | No | `None` |
| `simulator` | `boolean` | Se usare un simulatore o un QPU | No | `False` |
| `classical_optimizer` | `str` | Nome dell'ottimizzatore classico per il QAOA. Sono utilizzabili tutti i solver offerti da SciPy, elencati [qui](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize). Dovrai impostare `classical_optimizer_options` di conseguenza. | No | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opzioni dell'ottimizzatore classico. Per un elenco completo delle opzioni disponibili, consulta la [documentazione di SciPy](https://docs.scipy.org/doc/scipy/reference/) | No | `{"maxiter": 60}` |
| `optimization_level` | `integer` | La profondità del Circuit QAOA | No | `3` |
| `num_transpiler_runs` | `integer` | Numero di esecuzioni del Transpiler | No | `30` |
| `pass_manager_options` | `dictionary` | Opzioni per la generazione del preset pass manager | No | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opzioni dell'Estimator. Per un elenco completo delle opzioni disponibili, consulta la [documentazione del Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | No | `None` |
| `sampler_options` | `dictionary` | Opzioni del Sampler. Per un elenco completo delle opzioni disponibili, consulta la [documentazione del Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | No | `None` |

- Le `estimator_options` predefinite sono:

|     Nome    |    Tipo     | Valore  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Le `sampler_options` predefinite sono:

|     Nome    |    Tipo     | Valore |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Validazioni
- `name`:
    - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Non deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.

### 3. Delete
L'azione `delete` rimuove un classificatore dalla directory dati condivisa.

#### Input
|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Il nome dell'azione. Deve essere `delete`. | Sì |
| `name`      | `str`    | Il nome del classificatore da eliminare. | Sì |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Validazioni
- `name`:
    - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.

### 4. Fit
L'azione `fit` addestra un classificatore usando i dati di addestramento forniti.

#### Input
|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Il nome dell'azione. Deve essere `fit`. | Sì |
| `name`      | `str`    | Il nome del classificatore da addestrare. | Sì |
| `X`         | `array` o `list` o `str` | I dati di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `y`         | `array` o `list` o `str` | I valori target di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `fit_params`| `dictionary`| Parametri aggiuntivi da passare al metodo `fit` del classificatore. | No |

##### fit_params
|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |   Predefinito   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | I dati di validazione e le etichette. | No | `None` |
| `pos_label` | `integer` o `str` | L'etichetta di classe da mappare a 1. | No | `None` |
| `optimization_data` | `str` | Dataset su cui ottimizzare l'ensemble. Può essere uno tra: `train`, `validation`, `both`. | No | `train` |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Validazioni
- `name`:
    - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.

### 5. Predict
L'azione `predict` viene usata per ottenere previsioni hard e soft (probabilità).

#### Input
|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Il nome dell'azione. Deve essere `predict`. | Sì |
| `name`      | `str`    | Il nome del classificatore da utilizzare. | Sì |
| `X`         | `array` o `list` o `str` | I dati di test. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `options["out"]` | `str` | Il nome del file JSON di output per salvare le previsioni nella directory dati condivisa. Se non fornito, le previsioni vengono restituite nel risultato del job. | No |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Validazioni
- `name`:
    - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.
- `options["out"]`:
    - Il nome del file deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve avere l'estensione `.json`.

### 6. Fit-predict
L'azione `fit_predict` addestra un classificatore usando i dati di addestramento e poi lo usa per ottenere previsioni hard e soft (probabilità).

#### Input
|     Nome    |    Tipo     | Descrizione |   Obbligatorio  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Il nome dell'azione. Deve essere `fit_predict`. | Sì |
| `name`      | `str`    | Il nome del classificatore da utilizzare. | Sì |
| `X_train`   | `array` o `list` o `str` | I dati di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `y_train`   | `array` o `list` o `str` | I valori target di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `X_test`    | `array` o `list` o `str` | I dati di test. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `fit_params`| `dictionary`| Parametri aggiuntivi da passare al metodo `fit` del classificatore. | No |
| `options["out"]` | `str` | Il nome del file JSON di output per salvare le previsioni nella directory dati condivisa. Se non fornito, le previsioni vengono restituite nel risultato del job. | No |

#### Utilizzo

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Validazioni
- `name`:
    - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.

- `options["out"]`:
    - Il nome del file deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve avere l'estensione `.json`.

### 7. Create-fit-predict
L'azione `create_fit_predict` crea un classificatore, lo addestra usando i dati di addestramento forniti e poi lo usa per ottenere previsioni hard e soft (probabilità).

#### Input
| Nome | Tipo | Descrizione | Obbligatorio |
|------|------|-------------|-----------|
| `action` | `str` | Il nome dell'azione tra `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` e `delete`. | Sì |
| `name` | `str` | Il nome del classificatore da utilizzare. | Sì |
| `quantum_classifier` | `str` | Il tipo di classificatore, ovvero `QuantumEnhancedEnsembleClassifier`. Il valore predefinito è `QuantumEnhancedEnsembleClassifier`. | No |
| `X_train` | `array` o `list` o `str` | I dati di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `y_train` | `array` o `list` o `str` | I valori target di addestramento. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `X_test` | `array` o `list` o `str` | I dati di test. Può essere un array NumPy, una lista o una stringa che fa riferimento a un nome di file nella directory dati condivisa. | Sì |
| `fit_params` | `dictionary` | Parametri aggiuntivi da passare al metodo `fit` del classificatore. | No |
| `options["save"]` | `boolean` | Se salvare il classificatore addestrato nella directory dati condivisa. Il valore predefinito è `True`. | No |
| `options["out"]` | `str` | Il nome del file JSON di output per salvare le previsioni nella directory dati condivisa. Se non fornito, le previsioni vengono restituite nel risultato del job. | No |

#### Utilizzo

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Validazioni
- `name`:
    - Se `options["save"]` è impostato su `True`:
        - Il nome deve essere univoco, una stringa di massimo 64 caratteri.
        - Può contenere solo caratteri alfanumerici e underscore.
        - Deve iniziare con una lettera e non può terminare con un underscore.
        - Non deve esistere già un classificatore con lo stesso nome nella directory dati condivisa.

- `options["out"]`:
    - Il nome del file deve essere univoco, una stringa di massimo 64 caratteri.
    - Può contenere solo caratteri alfanumerici e underscore.
    - Deve iniziare con una lettera e non può terminare con un underscore.
    - Deve avere l'estensione `.json`.

---

## Per iniziare
Autenticati usando la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function come segue:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Esempio
In questo esempio, utilizzerai la funzione "Singularity Machine Learning - Classification" per classificare un dataset composto da due semicerchi a forma di mezza luna che si intrecciano. Il dataset è sintetico, bidimensionale ed etichettato con etichette binarie. È stato creato per essere difficile per algoritmi come il clustering basato su centroide e la classificazione lineare.
![Dataset moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Attraverso questo processo, imparerai come creare il classificatore, addestrarlo sui dati di addestramento, usarlo per effettuare previsioni sui dati di test ed eliminarlo al termine.
Prima di iniziare, devi installare [scikit-learn](https://scikit-learn.org/). Installalo usando il seguente comando:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Esegui i seguenti passaggi:

1) Crea il dataset sintetico usando la funzione [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) di [scikit-learn](https://scikit-learn.org/).
2) Carica il dataset sintetico generato nella [directory dati condivisa](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) Crea il classificatore potenziato quantisticamente usando l'azione [create](#2-create).
4) Elenca i tuoi classificatori usando l'azione [list](#1-list).
5) Addestra il classificatore sui dati di addestramento usando l'azione [fit](#4-fit).
6) Usa il classificatore addestrato per effettuare previsioni sui dati di test con l'azione [predict](#5-predict).
7) Elimina il classificatore usando l'azione [delete](#3-delete).
8) Fai pulizia quando hai finito.

**Passo 1.** Importa i moduli necessari e genera il dataset sintetico, poi suddividilo in dataset di addestramento e di test.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Passo 2.** Salva i dataset di addestramento e di test etichettati sul disco locale, quindi caricali nella [directory dati condivisa](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Passo 3.** Crea un classificatore potenziato quantisticamente usando l'azione [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>